### Stopping Conditions (Termination & Truncation) for CartPole-v1:

1. **Pole Angle Out of Bounds (Failure - `terminated = True`)**:
   - The pole angle exceeds $\pm 12$ degrees (approximately $\pm 0.209$ radians from the vertical).

2. **Cart Position Out of Bounds (Failure - `terminated = True`)**:
   - The cart center moves more than $\pm 2.4$ units away from the center of the track (it leaves the screen).

3. **Episode Length Reached (Success / Truncation - `truncated = True`)**:
   - The agent reaches the maximum number of allowed steps per episode (typically **500 steps** for `CartPole-v1`) without the pole falling.

In [61]:
from model import DQN
from replay_buffer import ReplayBuffer
import random
import torch
import torch.nn as nn
import torch.optim as optim
from time import sleep
import sys
import gymnasium as gym
import pygame
import torch.nn.functional as F

In [62]:
BATCH_SIZE = 64
GAMMA = 0.99  
EPS_START = 1.0  
EPS_END = 0.01  
EPS_DECAY = 0.994  
LR = 24e-5 
TARGET_UPDATE_K = 10 
NUM_EPISODES = 650

### Observations : x , $\dot{x}$ , $\theta$ , $\dot{\theta}$

In [63]:
env = gym.make("CartPole-v1")
n_observations = env.observation_space.shape[0]  #type:ignore
n_actions = env.action_space.n  #type:ignore
print("Observations: ",n_observations) 
print("Actions: ",n_actions)

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device :", device)



policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict()) 
optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)

print(f"Environnement ready")

Observations:  4
Actions:  2
device : cuda
Environnement ready


In [64]:
memory = ReplayBuffer(capacity=10000)
print("Replay Buffer ready !")

Replay Buffer ready !


In [65]:
def select_action(state, epsilon, n_actions):
  if random.random() < epsilon:
    # Exploration :
    return torch.tensor(
        [[env.action_space.sample()]], dtype=torch.long, device=device
    )
  else:
    # Exploitation :
    with torch.no_grad():
      return policy_net(state).max(1)[1].view(1, 1)

In [66]:
def optimize_model():
    # Only start training when the replay buffer contains enough samples
    if len(memory) < BATCH_SIZE:
        return

    # 1. Sample a random mini-batch from memory
    transitions = memory.sample(BATCH_SIZE)

    # Unpack the transitions (state, action, reward, next_state, terminated)
    batch_state, batch_action, batch_reward, batch_next_state, batch_terminated = (
        zip(*transitions)
    )

    # Convert data into PyTorch tensors
    batch_state = torch.cat(batch_state)
    batch_action = torch.cat(batch_action)
    batch_reward = torch.tensor(batch_reward, dtype=torch.float32, device=device).unsqueeze(1)
    batch_next_state = torch.cat(batch_next_state)
    batch_terminated = torch.tensor(
        batch_terminated, dtype=torch.float32, device=device
    ).unsqueeze(1)

    # 2. Calculate Q(s_t, a): the network's prediction for the actions actually taken
    state_action_values = policy_net(batch_state).gather(1, batch_action)

    # 3. Calculate V(s_{t+1}) for next states (using target_net for stability)
    with torch.no_grad():
        next_state_values = target_net(batch_next_state).max(1)[0].unsqueeze(1)

    # Bellman target: Q_target = reward + GAMMA * max(Q(s')) * (1 - terminated)
    expected_state_action_values = (
        batch_reward + (GAMMA * next_state_values * (1 - batch_terminated))
    )

    # 4. Calculate the loss between the prediction and the target
    criterion = nn.SmoothL1Loss()  # Huber Loss (robust to outliers)
    loss = criterion(state_action_values, expected_state_action_values)

    # 5. Backpropagation (update policy_net weights)
    optimizer.zero_grad()
    loss.backward()

    # Apply gradient clipping to prevent exploding gradients
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)

    optimizer.step()


print("select_action and optimize_model functions are ready!")

select_action and optimize_model functions are ready!


In [67]:
from tqdm import tqdm

epsilon = EPS_START

# Main loop with tqdm progress bar
pbar = tqdm(range(NUM_EPISODES), desc="DQN Training")
episode_rewards = []

for episode in pbar:
    observation, info = env.reset()
    terminated = False
    truncated = False

    # Convert initial observation into a tensor
    state = torch.tensor(
        observation, dtype=torch.float32, device=device
    ).unsqueeze(0)

    total_reward = 0
    steps_in_episode = 0

    while not (terminated or truncated):

        # 1. Select an action (Epsilon-greedy)
        action_tensor = select_action(state, epsilon, n_actions)
        action = action_tensor.item()

        # 2. Take a step in the environment
        next_observation, reward, terminated, truncated, info = env.step(action)
        total_reward += reward  # type: ignore
        steps_in_episode += 1

        # Convert next_state into a tensor
        next_state = torch.tensor(
            next_observation, dtype=torch.float32, device=device
        ).unsqueeze(0)
        done_flag = 1 if terminated else 0

        # 3. Store the transition in the Replay Buffer
        memory.push(state, action_tensor, reward, next_state, done_flag)

        # 4. Move to the next state
        state = next_state

        # 5. Perform training (Backpropagation on a mini-batch)
        optimize_model()

    # --- End of episode ---

    # Epsilon decay (gradually reduce exploration over time)
    epsilon = max(EPS_END, epsilon * EPS_DECAY)

    # Periodically update the Target Network (every TARGET_UPDATE_K episodes)
    if (episode + 1) % TARGET_UPDATE_K == 0:
        target_net.load_state_dict(policy_net.state_dict())

    episode_rewards.append(total_reward)

    # Update the tqdm progress bar display
    pbar.set_postfix(
        eps=f"{epsilon:.2f}",
        reward=total_reward,
        steps=steps_in_episode,
        buffer=len(memory),
    )

env.close()
print("Training completed successfully!")

DQN Training: 100%|██████████| 650/650 [02:27<00:00,  4.41it/s, buffer=1e+4, eps=0.02, reward=500, steps=500]

Training completed successfully!


In [68]:
print("mean of the last 50 total rewards: ",sum(episode_rewards[-50:]) / 50)

mean of the last 50 total rewards:  391.0


In [69]:
import os
import torch

os.makedirs("checkpoints", exist_ok=True)
save_path = "./checkpoints/dqn_cartpole_policy.pt"

checkpoint = {
    "model_state_dict": policy_net.state_dict(),
    "hyperparameters": {
        "BATCH_SIZE": BATCH_SIZE,
        "GAMMA": GAMMA,
        "EPS_START": EPS_START,
        "EPS_END": EPS_END,
        "EPS_DECAY": EPS_DECAY,
        "LR": LR,
        "TARGET_UPDATE_K": TARGET_UPDATE_K,
        "NUM_EPISODES": NUM_EPISODES,
    },
    "mean_last_50_reward": sum(episode_rewards[-50:]) / 50 if len(episode_rewards) >= 50 else 0,
}

torch.save(checkpoint, save_path)
print(f"saved in : {save_path}")


saved in : ./checkpoints/dqn_cartpole_policy.pt


In [70]:
import pygame
import torch

eval_env = gym.make("CartPole-v1", render_mode="human")

checkpoint = torch.load(
    "./checkpoints/dqn_cartpole_policy.pt", map_location=device
)

policy_net = DQN(n_observations, n_actions).to(device)
policy_net.load_state_dict(checkpoint["model_state_dict"])
policy_net.eval()

print("Mean last 50 reward:", checkpoint.get("mean_last_50_reward"))
print("Hyperparameters:", checkpoint.get("hyperparameters"))

NUM_TEST_EPISODES = 10
stop_requested = False

try:
  for episode in range(NUM_TEST_EPISODES):
    observation, info = eval_env.reset()
    terminated = False
    truncated = False
    total_reward = 0

    state = torch.tensor(
        observation, dtype=torch.float32, device=device
    ).unsqueeze(0)

    while not (terminated or truncated):
      # Gestion propre des événements clavier / fermeture
      if pygame.display.get_init():
        for event in pygame.event.get():
          if event.type == pygame.QUIT or (
              event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE
          ):
            stop_requested = True
            break

      if stop_requested:
        print("\nUser interrupt")
        break

      with torch.no_grad():
        action_tensor = policy_net(state).max(1)[1].view(1, 1)

      action = action_tensor.item()

      observation, reward, terminated, truncated, info = eval_env.step(action)
      total_reward += reward

      state = torch.tensor(
          observation, dtype=torch.float32, device=device
      ).unsqueeze(0)

    if stop_requested:
      break

    status = "truncated (500)" if truncated else "terminated"
    print(f"test episode {episode + 1} {status} | total rewards: {total_reward}")

finally:
  eval_env.close()
  if pygame.display.get_init():
    pygame.quit()

Mean last 50 reward: 391.0
Hyperparameters: {'BATCH_SIZE': 64, 'GAMMA': 0.99, 'EPS_START': 1.0, 'EPS_END': 0.01, 'EPS_DECAY': 0.994, 'LR': 0.00024, 'TARGET_UPDATE_K': 10, 'NUM_EPISODES': 650}
test episode 1 truncated (500) | total rewards: 500.0
test episode 2 truncated (500) | total rewards: 500.0
test episode 3 truncated (500) | total rewards: 500.0
test episode 4 truncated (500) | total rewards: 500.0
test episode 5 truncated (500) | total rewards: 500.0
test episode 6 truncated (500) | total rewards: 500.0
test episode 7 truncated (500) | total rewards: 500.0
test episode 8 truncated (500) | total rewards: 500.0
test episode 9 truncated (500) | total rewards: 500.0
test episode 10 truncated (500) | total rewards: 500.0
